# NB05 — Génération et Échantillonnage de Nanocorps
> **Phase E** : Stratégies de sampling (Greedy, Top-k, Nucleus/Top-p) et génération de 10 000 séquences par configuration.


In [ ]:
# ==============================================================================
# Bootstrap : détection du répertoire racine (local ET Google Colab)
# ==============================================================================
import sys
from pathlib import Path

def _find_root():
    """Walk up from cwd looking for utils.py (also checks <p>/project1/utils.py)."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for p in candidates:
        if (p / "utils.py").exists():
            return p
        if (p / "project1" / "utils.py").exists():
            # utils.py lives at repo root, not inside project1
            return p
    return None

ROOT_DIR = _find_root()

if ROOT_DIR is None:
    # ── Colab fallback: clone the repo ──────────────────────────────────────
    if Path("/content").exists():
        import subprocess
        clone_target = Path("/content/Recherche_AI")
        if not clone_target.exists():
            subprocess.run(
                ["git", "clone", "https://github.com/AhmadBelhaj4/Recherche_AI.git",
                 str(clone_target)],
                check=True,
            )
        ROOT_DIR = _find_root()

if ROOT_DIR is None:
    raise FileNotFoundError(
        "Could not locate utils.py. "
        "Set AI_PROJECT_DIR or run from inside the Recherche_AI repo."
    )

sys.path.insert(0, str(ROOT_DIR))
from utils import get_paths
PATHS = get_paths()
print(f"ROOT_DIR  : {ROOT_DIR}")
print(f"PATHS     : {PATHS}")


In [ ]:
# ==============================================================================
# Cellule 1 : Paramètres généraux (Settings) et chargement du modèle
# ==============================================================================
import sys
from pathlib import Path
import torch
import torch.nn.functional as F

from utils import set_seed, get_device, AminoAcidTokenizer
from models import CausalTransformerLM

SEED = 42
set_seed(SEED)
device = get_device()

N_SEQUENCES = 1000  # Changer à 10 000 pour la génération finale sur Colab
MAX_GEN_LEN = 140

tokenizer = AminoAcidTokenizer()
model = CausalTransformerLM(tokenizer.vocab_size).to(device)
model_path = PATHS["models"] / "transformer_best.pt"
if model_path.exists():
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Modèle chargé depuis {model_path}")
else:
    print("Modèle non pré-entraîné (mode démonstration avec poids initiaux).")
model.eval()


In [ ]:
# ==============================================================================
# Cellule 2 : Formules et fonctions de génération avec masquage exact d'arrêt
# ==============================================================================
def sample_next_token(logits, strategy="greedy", temperature=1.0, top_k=0, top_p=0.0):
    logits = logits / max(temperature, 1e-5)
    if strategy == "greedy":
        return torch.argmax(logits, dim=-1, keepdim=True)

    probs = F.softmax(logits, dim=-1)

    if top_k > 0:
        top_k = min(top_k, probs.size(-1))
        indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
        logits[indices_to_remove] = float('-inf')
        probs = F.softmax(logits, dim=-1)

    if top_p > 0.0:
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = False
        indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
        logits[indices_to_remove] = float('-inf')
        probs = F.softmax(logits, dim=-1)

    return torch.multinomial(probs, num_samples=1)

def generate_sequences(model, tokenizer, n_seqs=100, max_len=140, strategy="greedy", **kwargs):
    model.eval()
    generated = []
    batch_size = 64
    for start in range(0, n_seqs, batch_size):
        b = min(batch_size, n_seqs - start)
        tokens = torch.full((b, 1), tokenizer.bos_id, dtype=torch.long, device=device)
        finished = torch.zeros(b, dtype=torch.bool, device=device)

        with torch.no_grad():
            for _ in range(max_len):
                logits = model(tokens)[:, -1, :]
                next_tok = sample_next_token(logits, strategy=strategy, **kwargs)
                next_tok = torch.where(finished.unsqueeze(1), torch.tensor(tokenizer.pad_id, device=device), next_tok)
                tokens = torch.cat([tokens, next_tok], dim=1)
                finished = finished | (next_tok.squeeze(1) == tokenizer.eos_id)
                if finished.all():
                    break

        for seq_tokens in tokens.cpu().tolist():
            seq_str = tokenizer.decode(seq_tokens)
            generated.append(seq_str)
    return generated

print("Fonctions de génération configurées.")


In [ ]:
# ==============================================================================
# Cellule 3 : Génération des 4 configurations et sauvegarde en FASTA
# ==============================================================================
configs = [
    ("greedy", {"strategy": "greedy"}),
    ("topk10_T08", {"strategy": "top_k", "top_k": 10, "temperature": 0.8}),
    ("nucleus_p95_T10", {"strategy": "nucleus", "top_p": 0.95, "temperature": 1.0}),
    ("nucleus_p80_T12", {"strategy": "nucleus", "top_p": 0.80, "temperature": 1.2}),
]

gen_dir = PATHS["results"] / "generated"
gen_dir.mkdir(parents=True, exist_ok=True)

for name, cfg in configs:
    print(f"Génération : {name} ({N_SEQUENCES} séquences)...")
    seqs = generate_sequences(model, tokenizer, n_seqs=N_SEQUENCES, max_len=MAX_GEN_LEN, **cfg)
    fasta_path = gen_dir / f"{name}.fasta"
    with open(fasta_path, "w", encoding="utf-8") as f:
        for idx, s in enumerate(seqs):
            f.write(f">{name}_{idx}\n{s}\n")
    print(f"  Sauvegardé : {fasta_path}")
print("Toutes les configurations ont été générées avec succès.")
